In [ ]:
import json
import time
from pathlib import Path
from typing import List, Dict

import torch
import numpy as np
from tqdm import tqdm
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModelForTokenClassification

# ---------------- CONFIG ----------------

GOLD_FILE = "data/gold/gold.json"
SYNTH_FILE = "data/synthetic/synthetic.json"

STUDENT_MODEL_DIR = "models/student-ner"

OUTPUT_DIR = "evaluation_results"
STUDENT_PRED_FILE = f"{OUTPUT_DIR}/raw_student_model_labels.json"
METRICS_FILE = f"{OUTPUT_DIR}/metrics_summary.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- UTILS ----------------

def load_json(path: str) -> List[Dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def align_and_collect(y_true, y_pred):
    """
    Safety alignment: truncate to shortest sequence
    """
    aligned_true = []
    aligned_pred = []

    for t, p in zip(y_true, y_pred):
        min_len = min(len(t), len(p))
        aligned_true.append(t[:min_len])
        aligned_pred.append(p[:min_len])

    return aligned_true, aligned_pred


def evaluate_predictions(
    gold_data: List[Dict],
    pred_data: List[Dict],
    label: str
) -> Dict:
    """
    Compute Precision / Recall / F1
    """
    y_true = [s["ner_tags"] for s in gold_data]
    y_pred = [s["ner_tags"] for s in pred_data]

    y_true, y_pred = align_and_collect(y_true, y_pred)

    metrics = {
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred)
    }

    print(f"\n===== {label} Evaluation =====")
    print(classification_report(y_true, y_pred))
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall:    {metrics['recall']:.4f}")
    print(f"F1 Score:  {metrics['f1']:.4f}")

    return metrics

# ---------------- STUDENT MODEL PREDICTION ----------------

def predict_student_model(
    model,
    tokenizer,
    tokens: List[str]
) -> List[str]:
    """
    Token-level prediction aligned to words
    """
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )

    inputs = {k: v.to(DEVICE) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    preds = outputs.logits.argmax(dim=-1)[0].tolist()
    word_ids = encoding.word_ids(batch_index=0)

    aligned_preds = []
    prev_word_id = None

    for pred, word_id in zip(preds, word_ids):
        if word_id is None:
            continue
        if word_id != prev_word_id:
            aligned_preds.append(model.config.id2label[pred])
            prev_word_id = word_id

    return aligned_preds


def run_student_inference(
    gold_data: List[Dict]
) -> List[Dict]:
    """
    Run student model inference on gold tokens
    """
    tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_DIR)
    model = AutoModelForTokenClassification.from_pretrained(
        STUDENT_MODEL_DIR
    ).to(DEVICE)
    model.eval()

    predictions = []

    for sample in tqdm(gold_data, desc="Student model inference"):
        preds = predict_student_model(model, tokenizer, sample["tokens"])

        predictions.append({
            "id": sample["id"],
            "tokens": sample["tokens"],
            "ner_tags": preds
        })

    return predictions

# ---------------- MAIN ----------------

def main():
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    gold_data = load_json(GOLD_FILE)
    synthetic_data = load_json(SYNTH_FILE)

    # ---------------- LLM EVALUATION ----------------
    llm_metrics = evaluate_predictions(
        gold_data,
        synthetic_data,
        label="Mistral-7B-Instruct (LLM)"
    )

    # ---------------- STUDENT MODEL ----------------
    student_preds = run_student_inference(gold_data)

    with open(STUDENT_PRED_FILE, "w", encoding="utf-8") as f:
        json.dump(student_preds, f, indent=2)

    student_metrics = evaluate_predictions(
        gold_data,
        student_preds,
        label="Student Model (DistilBERT)"
    )

    # ---------------- SAVE METRICS ----------------
    summary = {
        "llm": llm_metrics,
        "student": student_metrics
    }

    with open(METRICS_FILE, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("\n Saved artifacts:")
    print(f" - Student predictions: {STUDENT_PRED_FILE}")
    print(f" - Metrics summary:     {METRICS_FILE}")

# ---------------- RUN ----------------

if __name__ == "__main__":
    main()